# Applying the Lagrange / KKT machinery to the live 12-position portfolio

Companion to `portfolio_lagrange_kkt.ipynb`. Same math, real holdings.

One thing changes and it changes everything: this portfolio is **long only**, so Part 3 (the closed-form
frontier with short sales) is the wrong tool. The relevant object is Part 5 — the quadratic program with
$X_i \ge 0$ — and its Kuhn-Tucker multipliers $\mu_i$, which say *how far each name is from being worth
holding at all*.

**Sections**
1. Positions → weights (from the broker screen, no market data needed)
2. Return data → covariance matrix (Ledoit-Wolf shrinkage)
3. Euler risk decomposition of the current weights
4. Long-only frontier, GMV and max-Sharpe portfolios
5. KKT multipliers: reduced costs and binding weight caps
6. One chart

**This notebook is analytics, not advice.** Optimizer weights are the output of a model whose inputs
(especially expected returns) are estimated with enormous error. Read $\mu_i$ as a diagnostic that tells you
*which assumption is doing the work*, not as an instruction.

## 1. Positions → weights

Quantities and prices as displayed. Nothing here needs a data feed — weights are just
$X_i = q_i p_i / \sum_j q_j p_j$, and they are the only portfolio input that is known exactly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.figsize": (9, 6), "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})

# ticker: (quantity, price) -- edit here when positions change
POSITIONS = {
    "MLI":  (4,  63.74),   "TTWO": (4,  214.69), "SCHO": (37, 24.02),
    "VTIP": (18, 49.73),   "META": (2,  616.77), "MMM":  (4,  168.56),
    "AVGO": (1,  357.90),  "VCSH": (6,  78.18),  "UL":   (8,  64.20),
    "PG":   (6,  146.44),  "NEE":  (5,  83.43),  "SPOT": (1,  542.43),
}

# sleeve labels, used only for reporting
SLEEVE = {"SCHO": "Short Treasury", "VTIP": "TIPS", "VCSH": "IG Credit",
          "META": "Comm Svcs", "TTWO": "Comm Svcs", "SPOT": "Comm Svcs",
          "AVGO": "Technology", "MMM": "Industrials", "MLI": "Industrials",
          "PG": "Staples", "UL": "Staples", "NEE": "Utilities"}

book = pd.DataFrame(POSITIONS, index=["qty", "price"]).T
book["value"] = book["qty"] * book["price"]
book["weight"] = book["value"] / book["value"].sum()
book["sleeve"] = pd.Series(SLEEVE)

tickers = list(book.index)
X_now = book["weight"].to_numpy()
n = len(tickers)

print(f"Total market value: {book['value'].sum():,.2f} USD across {n} positions\n")
print(book.sort_values("weight", ascending=False).round(4).to_string())

In [ ]:
# --- concentration and sleeve structure -------------------------------------
hhi = float((X_now ** 2).sum())
print(f"Herfindahl index      : {hhi:.4f}")
print(f"Effective N positions : {1/hhi:.2f} out of {n} held")
print(f"Largest position      : {book['weight'].max():.2%} ({book['weight'].idxmax()})\n")

by_sleeve = book.groupby("sleeve")["weight"].sum().sort_values(ascending=False)
print(by_sleeve.to_frame("weight").assign(pct=lambda d: (d["weight"] * 100).round(2)).to_string())

fixed_income = ["SCHO", "VTIP", "VCSH"]
w_fi = float(book.loc[fixed_income, "weight"].sum())
comm = ["META", "TTWO", "SPOT"]
w_comm = float(book.loc[comm, "weight"].sum())
print(f"\nFixed income sleeve   : {w_fi:.2%}")
print(f"Equity sleeve         : {1 - w_fi:.2%}")
print(f"Comm services         : {w_comm:.2%} of portfolio, {w_comm/(1-w_fi):.2%} of the equity sleeve")

## 2. Return data → covariance matrix

Everything downstream needs $\Sigma$. Two things worth being explicit about.

**Frequency.** Weekly returns over 3 years give ~156 observations for 12 assets. The sample covariance matrix
needs roughly $T \gg n$ to be well conditioned; at $T/n \approx 13$ it is estimable but noisy, and its extreme
eigenvalues are biased (largest too large, smallest too small). Daily data would give more observations but
imports microstructure noise and non-synchronous trading.

**Shrinkage.** Ledoit-Wolf pulls the sample covariance toward a structured target, trading a little bias for a
large variance reduction. This matters here specifically because the optimizer in section 4 inverts $\Sigma$,
and inversion amplifies exactly the smallest, worst-estimated eigenvalues.

Expected returns are the harder problem. Historical means over three years are close to useless as forecasts —
a 3-year sample gives a standard error on the mean of roughly $\sigma/\sqrt{3}$, so for a 25% vol stock that is
~14pp. The default below shrinks each name's historical mean halfway toward the cross-sectional average, which
is a crude James-Stein move. Override `MU_MANUAL` if you have real forecasts.

In [ ]:
LOOKBACK_YEARS = 3
FREQ = "W-FRI"          # weekly bars
PERIODS_PER_YEAR = 52
RF_ANNUAL = 0.042       # cash assumption, same as the CMA notebook
MU_SHRINK = 0.5         # 0 = raw historical mean, 1 = everyone gets the cross-sectional mean
MU_MANUAL = None        # e.g. pd.Series({"META": 0.09, ...}) to override with your own forecasts

def load_prices_yfinance(tickers, years):
    '''Adjusted close from Yahoo. Needs internet; returns None if unavailable.'''
    try:
        import yfinance as yf
    except ImportError:
        print("yfinance not installed -> pip install yfinance")
        return None
    end = pd.Timestamp.today().normalize()
    start = end - pd.DateOffset(years=years)
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    if raw is None or len(raw) == 0:
        print("download returned nothing")
        return None
    px = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw
    return px[tickers].dropna(how="all")

def load_prices_csv(path, tickers):
    '''Fallback: a CSV with a date index and one adjusted-close column per ticker.'''
    px = pd.read_csv(path, index_col=0, parse_dates=True)
    return px[tickers].dropna(how="all")

prices = load_prices_yfinance(tickers, LOOKBACK_YEARS)
# prices = load_prices_csv("prices.csv", tickers)   # <- use this if offline

assert prices is not None, "No price data. Install yfinance, or point load_prices_csv at a file."
rets = prices.resample(FREQ).last().pct_change().dropna()
print(f"{len(rets)} {FREQ} observations, {rets.index[0].date()} to {rets.index[-1].date()}")
print(f"T/n ratio: {len(rets)/n:.1f}")

In [ ]:
def covariance_matrix(returns, periods=PERIODS_PER_YEAR, shrink=True):
    '''Annualized covariance. Ledoit-Wolf shrinkage when sklearn is available.'''
    if shrink:
        try:
            from sklearn.covariance import LedoitWolf
            lw = LedoitWolf().fit(returns.to_numpy())
            print(f"Ledoit-Wolf shrinkage intensity: {lw.shrinkage_:.4f}")
            return pd.DataFrame(lw.covariance_ * periods, index=returns.columns, columns=returns.columns)
        except ImportError:
            print("sklearn not available -> sample covariance")
    return returns.cov() * periods

def expected_returns(returns, shrink=MU_SHRINK, manual=MU_MANUAL, periods=PERIODS_PER_YEAR):
    '''Annualized mean, shrunk toward the cross-sectional average.'''
    if manual is not None:
        return manual.reindex(returns.columns)
    raw = returns.mean() * periods
    return (1 - shrink) * raw + shrink * raw.mean()

Sigma_df = covariance_matrix(rets)
Sigma = Sigma_df.to_numpy()
Sigma = 0.5 * (Sigma + Sigma.T)
mu = expected_returns(rets)
R = mu.to_numpy()
sd_i = np.sqrt(np.diag(Sigma))
one = np.ones(n)

eig = np.linalg.eigvalsh(Sigma)
print(f"\nEigenvalues: min {eig.min():.6f}, max {eig.max():.6f}, condition number {eig.max()/eig.min():.1f}")
print(f"Positive definite: {bool((eig > 0).all())}\n")

print(pd.DataFrame({"E(R) shrunk": R, "raw hist mean": rets.mean() * PERIODS_PER_YEAR,
                    "sigma": sd_i, "weight": X_now}, index=tickers).sort_values("weight",
                    ascending=False).round(4).to_string())

## 3. Euler risk decomposition of the current weights

This is the same object as the first-order condition from the Lagrangian. Since $\sigma_p = \sqrt{X'\Sigma X}$
is homogeneous of degree 1 in $X$, Euler's theorem gives an exact additive split:

$$\sigma_p = \sum_i X_i \cdot \underbrace{\frac{(\Sigma X)_i}{\sigma_p}}_{\text{MCR}_i}$$

$\text{MCR}_i$ is the derivative $\partial\sigma_p/\partial X_i$ — the risk added by one more unit of asset $i$,
*given everything else already held*. It is not asset $i$'s standalone volatility, and the gap between the two
is entirely the correlation story. A name can be individually volatile and still contribute little, or be
individually calm and contribute a lot because it duplicates the rest of the book.

At the optimum of the Part 3 problem, $\Sigma X = \lambda_1 R + \lambda_2 \mathbf{1}$, so risk contributions
line up linearly with expected returns. Deviations from that line are exactly what the optimizer would fix.

In [ ]:
def risk_decomposition(X, Sigma, R, labels):
    '''Euler decomposition: MCR, component contribution, share of total risk.'''
    sd_p = float(np.sqrt(X @ Sigma @ X))
    mcr = (Sigma @ X) / sd_p
    ccr = X * mcr
    out = pd.DataFrame({"weight": X, "sigma_i": np.sqrt(np.diag(Sigma)), "MCR": mcr,
                        "risk contrib": ccr, "% of risk": ccr / sd_p,
                        "risk / weight": (ccr / sd_p) / X, "E(R)": R}, index=labels)
    return out, sd_p

decomp, sd_now = risk_decomposition(X_now, Sigma, R, tickers)
R_now = float(X_now @ R)

print(f"Portfolio E(R) = {R_now:.4%}   sigma = {sd_now:.4%}   "
      f"Sharpe = {(R_now - RF_ANNUAL)/sd_now:.4f}\n")
print(decomp.sort_values("% of risk", ascending=False).round(4).to_string())
print(f"\nCheck: risk contributions sum to sigma_p -> {decomp['risk contrib'].sum():.6f} vs {sd_now:.6f}")
print(f"Diversification ratio (weighted avg vol / portfolio vol): "
      f"{float(X_now @ np.sqrt(np.diag(Sigma))) / sd_now:.3f}")

## 4. Long-only frontier, GMV and max-Sharpe

Now the constrained problem from Part 5, on real holdings:

$$\min_X \tfrac{1}{2}X'\Sigma X \quad \text{s.t.}\quad X'R = R_p^*,\ X'\mathbf{1}=1,\ 0 \le X_i \le \text{cap}$$

The weight cap is added because a 12-name book with no cap will happily put 60% into whichever bond ETF has
the lowest estimated variance. The cap is a statement about estimation error, not about the math.

In [ ]:
MAX_WEIGHT = 0.25       # position cap; set to 1.0 to disable
TOL = 1e-7

def variance_obj(X):
    return 0.5 * X @ Sigma @ X

def variance_grad(X):
    return Sigma @ X

def budget_con(X):
    return X.sum() - 1.0

def budget_jac(X):
    return one

def return_con(X, target):
    return X @ R - target

def return_jac(X, target):
    return R

def solve_long_only(target=None, x0=None, cap=MAX_WEIGHT):
    '''Min variance subject to budget, optional target return, and 0 <= X_i <= cap.'''
    cons = [{"type": "eq", "fun": budget_con, "jac": budget_jac}]
    if target is not None:
        cons.append({"type": "eq", "fun": return_con, "jac": return_jac, "args": (target,)})
    if x0 is None:
        x0 = np.full(n, 1.0 / n)
    return minimize(variance_obj, x0, method="SLSQP", jac=variance_grad,
                    bounds=[(0.0, cap)] * n, constraints=cons,
                    options={"maxiter": 800, "ftol": 1e-14})

def negative_sharpe(X):
    return -(X @ R - RF_ANNUAL) / np.sqrt(X @ Sigma @ X)

def solve_max_sharpe(cap=MAX_WEIGHT):
    return minimize(negative_sharpe, np.full(n, 1.0 / n), method="SLSQP",
                    bounds=[(0.0, cap)] * n,
                    constraints=[{"type": "eq", "fun": budget_con, "jac": budget_jac}],
                    options={"maxiter": 800, "ftol": 1e-12})

X_gmv = solve_long_only().x
X_tan = solve_max_sharpe().x

summary = pd.DataFrame({"current": X_now, "GMV long-only": X_gmv, "max-Sharpe long-only": X_tan},
                       index=tickers)
stats = pd.DataFrame({
    "E(R)":   [X_now @ R, X_gmv @ R, X_tan @ R],
    "sigma":  [np.sqrt(X_now @ Sigma @ X_now), np.sqrt(X_gmv @ Sigma @ X_gmv),
               np.sqrt(X_tan @ Sigma @ X_tan)]}, index=summary.columns)
stats["Sharpe"] = (stats["E(R)"] - RF_ANNUAL) / stats["sigma"]

print(summary.round(4).to_string()); print()
print(stats.round(4).to_string())
print(f"\nRisk you could remove at the same E(R): "
      f"{1e4*(np.sqrt(X_now @ Sigma @ X_now) - np.sqrt(solve_long_only(R_now).x @ Sigma @ solve_long_only(R_now).x)):.0f} bp")

In [ ]:
# --- trace the long-only efficient frontier ---------------------------------
def attainable_range(R, cap):
    '''Highest / lowest E(R) reachable with 0 <= X_i <= cap and sum(X) = 1:
    fill the cap greedily into the best (worst) names.'''
    m = int(np.floor(1.0 / cap))
    fill = np.zeros(len(R))
    fill[:m] = cap
    if m < len(R):
        fill[m] = 1.0 - m * cap
    return float(fill @ np.sort(R)[::-1]), float(fill @ np.sort(R))

R_max_cap, R_min_cap = attainable_range(R, MAX_WEIGHT)
R_gmv = float(X_gmv @ R)
print(f"With a {MAX_WEIGHT:.0%} cap, attainable E(R) spans {R_min_cap:.4%} to {R_max_cap:.4%}")
targets = np.linspace(R_gmv, R_max_cap - 1e-6, 40)

sd_frontier, warm = [], X_gmv.copy()
for t in targets:                       # each target is its own QP; warm start the next
    res = solve_long_only(t, x0=warm)
    if res.status == 0:
        warm = res.x
        sd_frontier.append(float(np.sqrt(res.x @ Sigma @ res.x)))
    else:
        sd_frontier.append(np.nan)
sd_frontier = np.array(sd_frontier)
print(f"Frontier solved at {np.isfinite(sd_frontier).sum()} of {len(targets)} targets")
print(f"Feasible return range under the cap: {R_gmv:.4%} to {targets[np.isfinite(sd_frontier)].max():.4%}")

## 5. KKT multipliers: what the constraints cost

With bounds on both sides the Lagrangian carries two sets of multipliers:

$$L = \tfrac{1}{2}X'\Sigma X - \lambda_1(X'R - R_p^*) - \lambda_2(X'\mathbf{1}-1)
- \sum_i \mu_i X_i + \sum_i \nu_i (X_i - \text{cap})$$

Stationarity gives $(\Sigma X)_i - \lambda_1 R_i - \lambda_2 - \mu_i + \nu_i = 0$. Define

$$g_i = (\Sigma X)_i - \lambda_1 R_i - \lambda_2$$

and complementary slackness reads it off directly:

| position of $X_i$ | multipliers | sign of $g_i$ | meaning |
|---|---|---|---|
| interior | $\mu_i = \nu_i = 0$ | $g_i = 0$ | the equality-constrained FOC holds |
| at zero | $\mu_i \ge 0$, $\nu_i = 0$ | $g_i \ge 0$ | **reduced cost**: how much the name is "priced out" |
| at cap | $\mu_i = 0$, $\nu_i \ge 0$ | $g_i \le 0$ | the cap is binding and costing you |

The reduced cost is the useful number. It ranks the excluded names by how close they are to entering, and it
converts directly into a required-return statement: raising $R_i$ by $\mu_i/\lambda_1$ is exactly what it would
take for name $i$ to enter the optimal portfolio. That is a falsifiable claim about your own assumptions.

In [ ]:
def kkt_report(X, Sigma, R, labels, cap=MAX_WEIGHT, tol=TOL):
    '''Recover lambda_1, lambda_2 from the free set, then g_i for every asset.'''
    free = (X > tol) & (X < cap - tol)
    if free.sum() < 2:
        print("Fewer than two interior assets; multipliers not identified.")
        return None, None, None
    M = np.column_stack([R[free], np.ones(int(free.sum()))])
    (l1, l2), *_ = np.linalg.lstsq(M, (Sigma @ X)[free], rcond=None)
    g = Sigma @ X - l1 * R - l2

    state = np.where(X <= tol, "at zero", np.where(X >= cap - tol, "at cap", "interior"))
    # "required return to enter" only makes sense on the efficient branch, where l1 > 0
    if l1 > 1e-12:
        req = np.where(X <= tol, g / l1, np.nan)
    else:
        print("lambda_1 <= 0: this target sits on the inefficient branch, "
              "so required-return figures are not meaningful. Use a target above the GMV return.")
        req = np.full(len(X), np.nan)
    table = pd.DataFrame({"weight": X, "state": state, "g_i (mu or -nu)": g,
                          "E(R) used": R, "required E(R) to enter": R + req}, index=labels)
    return l1, l2, table

if R_now < R_gmv:
    print(f"NOTE: the book's E(R) ({R_now:.4%}) is below the long-only GMV return ({R_gmv:.4%}),\n"
          f"      so the same-return optimum lies on the inefficient branch. "
          f"Comparing against GMV instead is more informative here.\n")

X_star = solve_long_only(R_now).x            # same expected return as the current book
X_star = np.where(X_star < TOL, 0.0, X_star)
l1, l2, kkt = kkt_report(X_star, Sigma, R, tickers)

print(f"Optimal long-only portfolio at the current E(R) = {R_now:.4%}")
print(f"lambda_1 = {l1:.6f}  (marginal variance cost of one more unit of return)")
print(f"lambda_2 = {l2:.6f}\n")
print(kkt.sort_values("g_i (mu or -nu)", ascending=False).round(4).to_string())

excluded = kkt[kkt["state"] == "at zero"]
print(f"\n{len(excluded)} of {n} names sit at the zero bound in the optimum.")
print("Ranked by reduced cost, i.e. how far each is from being worth holding under THESE inputs:")
print(excluded.sort_values("g_i (mu or -nu)", ascending=False)[
      ["g_i (mu or -nu)", "E(R) used", "required E(R) to enter"]].round(4).to_string())

## 6. The picture

In [ ]:
fig, ax = plt.subplots()

ok = np.isfinite(sd_frontier)
ax.plot(sd_frontier[ok], targets[ok], color="tab:purple", lw=2.2,
        label=f"long-only frontier (cap {MAX_WEIGHT:.0%})")
ax.scatter([sd_now], [R_now], color="tab:red", s=110, marker="*", zorder=6, label="current portfolio")
ax.scatter([np.sqrt(X_gmv @ Sigma @ X_gmv)], [X_gmv @ R], color="tab:purple", marker="s",
           s=55, zorder=6, label="GMV long-only")
ax.scatter([np.sqrt(X_tan @ Sigma @ X_tan)], [X_tan @ R], color="tab:orange", s=80,
           zorder=6, label="max-Sharpe long-only")
ax.scatter(sd_i, R, color="black", marker="D", s=28, zorder=5, label="individual holdings")
for name, s, r in zip(tickers, sd_i, R):
    ax.annotate(name, (s, r), textcoords="offset points", xytext=(7, -3), fontsize=8)

sd_line = np.linspace(0, sd_i.max() * 1.05, 50)
sharpe_tan = (X_tan @ R - RF_ANNUAL) / np.sqrt(X_tan @ Sigma @ X_tan)
ax.plot(sd_line, RF_ANNUAL + sharpe_tan * sd_line, color="tab:green", lw=1.5, ls="--",
        label="CML through max-Sharpe")

ax.set_xlabel(r"annualized $\sigma_p$"); ax.set_ylabel(r"annualized $E(R_p)$")
ax.set_title("Current book against its own long-only frontier")
ax.legend(loc="best", fontsize=9); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
order = np.argsort(-decomp["% of risk"].to_numpy())
idx = np.arange(n)
ax.bar(idx - 0.2, decomp["weight"].to_numpy()[order], width=0.4, label="weight")
ax.bar(idx + 0.2, decomp["% of risk"].to_numpy()[order], width=0.4, label="% of risk")
ax.set_xticks(idx); ax.set_xticklabels(np.array(tickers)[order], rotation=45, ha="right")
ax.set_title("Where the money sits vs where the risk sits"); ax.legend(); plt.show()

## How to read the output

**Weights are exact, everything else is estimated.** Section 1 needs no assumptions. Section 3 needs $\Sigma$
only, which is the best-estimated input. Sections 4 and 5 need $R$, which is the worst. So the ordering of
trust is: concentration stats > risk decomposition > optimal weights.

**The gap between weight and risk share is the finding.** A name whose "% of risk" badly exceeds its weight is
carrying the book's volatility; a bond ETF with 11% weight and 1% of risk is doing the opposite. That
comparison holds up even though the expected returns are guesswork, which is why the second chart is arguably
more useful than the first.

**Reduced costs are a test of your own assumptions, not a signal.** If a name needs +4pp of expected return
before the optimizer would hold it, the question to ask is whether your thesis for it actually implies 4pp of
outperformance. If it does, the input is wrong, not the position. If it doesn't, the position rests on
something outside the mean-variance frame — which is fine, but should be said out loud in the risk note.

**Known limitations of this setup.** Three years of weekly data cannot see a regime change; the covariance
matrix has no factor structure, so sector clustering is only implicit; the shrinkage on expected returns is
crude compared with Black-Litterman; and the whole exercise is single-period, so nothing here says anything
about rebalancing costs or turnover.